In [1]:
import json
import os

# We need to set these environment variables before importing neuron_explainer
os.environ["NEURON_EXPLAINER_API_KEY"] = "EMPTY"
os.environ["NEURON_EXPLAINER_API_BASE"] = "http://localhost:8080/v1"

from neuron_explainer.activations.activation_records import calculate_max_activation
from neuron_explainer.activations.activations import (
    ActivationRecordSliceParams,
)
from neuron_explainer.explanations.calibrated_simulator import (
    UncalibratedNeuronSimulator,
)
from neuron_explainer.explanations.explainer import TokenActivationPairExplainer
from neuron_explainer.explanations.prompt_builder import PromptFormat
from neuron_explainer.explanations.scoring import (
    simulate_and_score,
    aggregate_scored_sequence_simulations,
)
from neuron_explainer.explanations.simulator import ExplanationNeuronSimulator
from neuron_explainer.fast_dataclasses.fast_dataclasses import loads
from pathlib import Path


In [6]:
from html import escape
from IPython.display import HTML, display


def show_token_activations(record):
    tokens = record["tokens"]
    activations = [max(0.0, x) for x in record["activations"]]
    peak = max(activations) or 1.0
    spans = []
    for token, activation in zip(tokens, activations):
        intensity = activation / peak
        color = f"rgba(0, 160, 0, {0.15 + 0.85 * intensity})"
        spans.append(
            f"<span style='background:{color}; padding:2px 4px; margin:1px; "
            f"border-radius:3px; display:inline-block;'>{escape(token)}</span>"
        )
    html = "<div style='font-family:monospace; line-height:2;'>" + " ".join(spans) + "</div>"
    display(HTML(html))


# Example usage (replace with real tokens/activations from your recorder)
demo_record = {
    "tokens": ["The", "quick", "brown", "fox"],
    "activations": [0.1, 0.8, 0.4, -0.2],
}
show_token_activations(demo_record)


In [50]:
def sanitize_tokens(tokens):
    new_tokens = []
    for token in tokens:
        new_tokens.append(token.replace("Ġ", "").replace("Ċ", ""))
    return new_tokens


In [102]:
path = Path("../../../activations/slice_processed_29/26772.blob")
with open(path, "rb") as f:
    feature_record = loads(f.read())
for i in range(len(feature_record.random_sample)):
    feature_record.random_sample[i].tokens = sanitize_tokens(
        feature_record.random_sample[i].tokens
    )

for i in range(len(feature_record.most_positive_activation_records)):
    feature_record.most_positive_activation_records[i].tokens = sanitize_tokens(
        feature_record.most_positive_activation_records[i].tokens
    )

In [103]:
for i in range(5):
    record = {
        "tokens": feature_record.most_positive_activation_records[i].tokens,
        "activations": feature_record.most_positive_activation_records[i].activations,
    }

    show_token_activations(record)
